# DLGenAI Project 26T3 — Milestone 4 (Google Colab Solution Notebook)


## What it solves

- **Q1–Q3:** image/class imbalance, defect_b pixel prevalence, and bounded square-root sampling on the first 1,200 image IDs.
- **Q4–Q7:** exact scheduler, early-stopping, memory, and official-metric calculations.
- **Q8:** six-epoch baseline training with per-epoch train loss, validation loss, and official validation score logged to W&B.
- **Q9:** defect-size bins and expected missed-pair analysis.
- **Q10:** four controlled OFAT runs (R0, R0b, R1, R2), each six epochs, each logged to W&B.

## Important setup notes

1. This follows: first 1,200 unique `image_id` values after ASCII/alphabetical sorting; four class rows per image; native 128×800 masks; clipped 1-based Fortran-order RLE; 80/20 stratified split on combination strings after merging groups with fewer than 3 images into `rare_other`; `random_state=42`.
2. The assignment says use the **same model as the Milestone 2/3 baseline**. This notebook defaults to the project's U-Net with ResNet34 encoder. Confirm that matches your M2/M3 notebook. If that baseline used a different architecture or loss, change the clearly marked `BASELINE MODEL CONFIG` cell before training so Q8/Q10 are faithful to your earlier baseline.
3. Training uses batch size 16, AdamW, base LR 3e-4, AMP when CUDA is available, seed 42, threshold 0.5 and no minimum-area filtering. The PDF's optimizer-step scheduler is applied per optimizer update (not once per epoch).
4. W&B logging is required by the PDF. Add `WANDB_API_KEY` to Colab Secrets (recommended). The notebook uses the existing IITM Forge W&B entity/project settings from the project history.


The notebook prints the answer values in the requested formats. Data-dependent numbers are calculated from the official competition CSV and masks when you run it; they are not fabricated in advance.

## 0. Install required packages

In [1]:
%pip -q install kaggle segmentation-models-pytorch wandb scikit-learn pillow tqdm

## 1. Connect Google Drive and configure persistent outputs

In [2]:
from pathlib import Path
import os, json, shutil, subprocess, zipfile, math, random, time, csv, warnings, getpass
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from PIL import Image

from google.colab import drive, files, userdata

drive.mount("/content/drive")

SEED = 42
CLASSES = ["defect_a", "defect_b", "defect_c", "defect_d"]
N_CLASSES = len(CLASSES)
HEIGHT, WIDTH = 128, 800
N_PIXELS = HEIGHT * WIDTH
WORK_DIR = Path("/content/dlgenai_milestone4")
DATA_DIR = WORK_DIR / "competition_data"
EXTRACT_DIR = DATA_DIR / "extracted"
ARTIFACT_DIR = Path("/content/drive/MyDrive/DLGenAI_Milestone4")
for p in [WORK_DIR, DATA_DIR, EXTRACT_DIR, ARTIFACT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("Persistent output directory:", ARTIFACT_DIR)
print("Runtime:", "CUDA available" if __import__("torch").cuda.is_available() else "CPU only")

Mounted at /content/drive
Persistent output directory: /content/drive/MyDrive/DLGenAI_Milestone4
Runtime: CUDA available


## 2. Kaggle authentication and competition-data download

In [3]:
# Uses MyDrive/kaggle.json when present; otherwise securely prompts for an upload.
KAGGLE_JSON = Path("/content/drive/MyDrive/kaggle.json")
KAGGLE_DIR = Path.home() / ".kaggle"
KAGGLE_DIR.mkdir(parents=True, exist_ok=True)
TARGET_KAGGLE_JSON = KAGGLE_DIR / "kaggle.json"

if KAGGLE_JSON.is_file():
    shutil.copy2(KAGGLE_JSON, TARGET_KAGGLE_JSON)
else:
    print("Upload your kaggle.json file. Its contents will not be printed.")
    uploaded = files.upload()
    json_names = [name for name in uploaded if name.lower().endswith(".json")]
    if len(json_names) != 1:
        raise RuntimeError("Upload exactly one Kaggle API token file named kaggle.json.")
    TARGET_KAGGLE_JSON.write_bytes(uploaded[json_names[0]])

os.chmod(TARGET_KAGGLE_JSON, 0o600)

COMPETITION_SLUG = "industrial-surface-defect-detection"
cmd = ["kaggle", "competitions", "download", "-c", COMPETITION_SLUG, "-p", str(DATA_DIR), "--force"]
proc = subprocess.run(cmd, capture_output=True, text=True)
print(proc.stdout[-2000:])
if proc.returncode != 0:
    raise RuntimeError(
        "Kaggle download failed. Check that competition rules are accepted and kaggle.json "
        "belongs to an account with access. Kaggle error:\n" + proc.stderr[-3000:]
    )

zip_files = list(DATA_DIR.glob("*.zip"))
if not zip_files:
    raise FileNotFoundError(f"No competition ZIP was downloaded into {DATA_DIR}.")
for zpath in zip_files:
    with zipfile.ZipFile(zpath) as zf:
        zf.extractall(EXTRACT_DIR)

candidates = []
for csv_path in EXTRACT_DIR.rglob("train.csv"):
    root = csv_path.parent
    if (root / "train").is_dir():
        candidates.append(root)
if len(candidates) != 1:
    raise RuntimeError(f"Expected one extracted competition public folder; found: {candidates}")
DATA_ROOT = candidates[0]
TRAIN_CSV = DATA_ROOT / "train.csv"
TRAIN_IMAGE_DIR = DATA_ROOT / "train"
print("Competition data root:", DATA_ROOT)
print("Training images found:", len(list(TRAIN_IMAGE_DIR.glob("*.jpg"))))

Upload your kaggle.json file. Its contents will not be printed.


Saving kaggle.json to kaggle.json


Competition data root: /content/dlgenai_milestone4/competition_data/extracted/public
Training images found: 11393


## 3. Imports, deterministic setup and RLE decoding

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import segmentation_models_pytorch as smp
from sklearn.model_selection import train_test_split
from contextlib import nullcontext

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_ENABLED = DEVICE.type == "cuda"
BATCH_SIZE = 16
BASE_LR = 3e-4
EPOCHS = 6
DECISION_THRESHOLD = 0.5


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    # Reproducibility is preferred for these fixed-budget experiments.
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

set_seed(SEED)
print("Device:", DEVICE, "| AMP:", AMP_ENABLED)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


def decode_rle_clipped(rle, height=HEIGHT, width=WIDTH):
    """Decode 1-based Fortran-order RLE, clipping runs to the image canvas."""
    flat = np.zeros(height * width, dtype=np.uint8)
    if rle is None or (isinstance(rle, float) and np.isnan(rle)) or str(rle).strip() == "":
        return flat.reshape((height, width), order="F")
    values = np.asarray(str(rle).split(), dtype=np.int64)
    if len(values) % 2:
        raise ValueError(f"Malformed RLE start-length sequence: {rle!r}")
    starts = values[0::2] - 1
    lengths = values[1::2]
    for start, length in zip(starts, lengths):
        if start < 0 or length <= 0:
            raise ValueError(f"Invalid RLE start/length: {start+1}, {length}")
        end = min(int(start + length), height * width)
        if start < height * width and end > start:
            flat[int(start):end] = 1
    return flat.reshape((height, width), order="F")


def encode_rle_column_major(mask):
    pixels = np.asarray(mask, dtype=np.uint8).reshape(-1, order="F")
    if not np.any(pixels):
        return " "
    padded = np.concatenate(([0], pixels, [0])).astype(np.uint8)
    changes = np.flatnonzero(padded[1:] != padded[:-1]) + 1
    starts, ends = changes[0::2], changes[1::2]
    values = np.empty(starts.size * 2, dtype=np.int64)
    values[0::2] = starts
    values[1::2] = ends - starts
    return " ".join(map(str, values.tolist()))

# Native-canvas decode/encode test, including an empty mask.
_toy = np.zeros((3, 4), dtype=np.uint8)
_toy[0, 0] = _toy[1, 0] = _toy[2, 2] = 1
assert encode_rle_column_major(_toy) == "1 2 9 1"
assert np.array_equal(decode_rle_clipped("1 2 9 1", 3, 4), _toy)
assert encode_rle_column_major(np.zeros((3, 4), dtype=np.uint8)) == " "
print("RLE decode/encode tests: PASSED")

Device: cuda | AMP: True
GPU: Tesla T4
RLE decode/encode tests: PASSED


## 4. Load the exact 1,200-image working subset and decode masks

In [5]:
train_df = pd.read_csv(TRAIN_CSV, dtype={"image_id": str, "defect_type": str})
required = {"image_id", "defect_type", "mask_rle"}
if not required.issubset(train_df.columns):
    raise ValueError(f"train.csv missing columns: {required - set(train_df.columns)}")
train_df["image_id"] = train_df["image_id"].astype(str)
train_df["defect_type"] = train_df["defect_type"].astype(str)
if not set(train_df["defect_type"].unique()).issubset(set(CLASSES)):
    raise ValueError("Unexpected defect_type value in train.csv")

# ASCII sorting is Python's default lexicographic order for these string IDs.
all_image_ids_sorted = sorted(train_df["image_id"].dropna().unique().tolist())
if len(all_image_ids_sorted) < 1200:
    raise RuntimeError(f"Need at least 1,200 unique images; found {len(all_image_ids_sorted)}")
WORKING_IDS = all_image_ids_sorted[:1200]
work_df = train_df[train_df["image_id"].isin(WORKING_IDS)].copy()

# Confirm exactly one row for each image/class pair before treating null mask_rle as clean.
if work_df.duplicated(subset=["image_id", "defect_type"]).any():
    raise ValueError("Duplicate image_id/defect_type rows in the 1,200-image subset")
expected_keys = {(i, c) for i in WORKING_IDS for c in CLASSES}
actual_keys = set(zip(work_df["image_id"], work_df["defect_type"]))
if actual_keys != expected_keys:
    raise ValueError(
        f"Subset does not contain exactly four class rows per image. Missing={len(expected_keys-actual_keys)}, "
        f"extra={len(actual_keys-expected_keys)}"
    )

rle_pivot = work_df.pivot(index="image_id", columns="defect_type", values="mask_rle").reindex(
    index=WORKING_IDS, columns=CLASSES
)
# After key-coverage validation, null mask values represent empty/clean masks.
rle_pivot = rle_pivot.fillna("")
MASKS = np.zeros((1200, N_CLASSES, HEIGHT, WIDTH), dtype=np.uint8)
for i, image_id in enumerate(tqdm(WORKING_IDS, desc="Decoding 1,200 × 4 masks")):
    for c_idx, class_name in enumerate(CLASSES):
        MASKS[i, c_idx] = decode_rle_clipped(rle_pivot.loc[image_id, class_name])

AREA = MASKS.sum(axis=(2, 3), dtype=np.uint32)  # (1200, 4)
PRESENCE = AREA > 0

def combination_string(row):
    present = [CLASSES[j] for j, value in enumerate(row) if value]
    return "+".join(present) if present else "clean"

COMBINATIONS = np.asarray([combination_string(row) for row in PRESENCE], dtype=object)
combo_counts = pd.Series(COMBINATIONS).value_counts().sort_index()
print("Working subset images:", len(WORKING_IDS))
print("Image-class pairs:", len(WORKING_IDS) * N_CLASSES)
print("Decoded mask tensor:", MASKS.shape, MASKS.nbytes / (1024**2), "MiB")
print("Raw combination groups:", len(combo_counts))
display(combo_counts.rename("images").to_frame())

Decoding 1,200 × 4 masks:   0%|          | 0/1200 [00:00<?, ?it/s]

Working subset images: 1200
Image-class pairs: 4800
Decoded mask tensor: (1200, 4, 128, 800) 468.75 MiB
Raw combination groups: 15


,images
clean,578
defect_a,59
defect_a+defect_b,2
defect_a+defect_b+defect_c,1
defect_a+defect_b+defect_d,3
defect_a+defect_c,21
defect_a+defect_c+defect_d,7
defect_a+defect_d,14
defect_b,22
defect_b+defect_c,2


## 5. Q1 — Image-level and class-level imbalance; exact 960/240 split

In [6]:
clean_count = int(np.sum(COMBINATIONS == "clean"))
positive_by_class = {c: int(PRESENCE[:, j].sum()) for j, c in enumerate(CLASSES)}
raw_group_sizes = pd.Series(COMBINATIONS).value_counts()
q1_unique_combinations = int(len(raw_group_sizes))
q1_largest_name = str(raw_group_sizes.idxmax())
q1_smallest_name = str(raw_group_sizes.idxmin())
q1_largest_size = int(raw_group_sizes.max())
q1_smallest_size = int(raw_group_sizes.min())

# Merge every raw combination with fewer than 3 images into the single rare_other stratum.
rare_groups = set(raw_group_sizes[raw_group_sizes < 3].index.tolist())
STRATIFY_LABELS = np.asarray([
    "rare_other" if value in rare_groups else value for value in COMBINATIONS
], dtype=object)
strata_counts = pd.Series(STRATIFY_LABELS).value_counts()
if int(strata_counts.min()) < 2:
    raise RuntimeError(f"A post-merge stratum still has fewer than 2 images: {strata_counts.to_dict()}")

train_ids, val_ids = train_test_split(
    np.asarray(WORKING_IDS, dtype=str), test_size=0.20,
    random_state=42, stratify=STRATIFY_LABELS
)
train_ids, val_ids = list(train_ids), list(val_ids)
assert len(train_ids) == 960 and len(val_ids) == 240
assert not (set(train_ids) & set(val_ids))

print("Q1(a) Completely clean images:", clean_count)
print("Q1(b) Positive images [defect_a, defect_b, defect_c, defect_d]:",
      [positive_by_class[c] for c in CLASSES])
print("Q1(c) Unique raw combination strings (before merging):", q1_unique_combinations)
print("Q1(d) Largest raw group:", q1_largest_name, q1_largest_size)
print("Q1(d) Smallest raw group:", q1_smallest_name, q1_smallest_size)
print("Rare groups merged into rare_other:", sorted(rare_groups))
merged_largest_group = str(strata_counts.idxmax())
merged_smallest_group = str(strata_counts.idxmin())
merged_largest_size = int(strata_counts.max())
merged_smallest_size = int(strata_counts.min())
print("Split groups after merging:\n", strata_counts.to_dict())
print("Merged-group extrema (additional diagnostic): largest", merged_largest_group, merged_largest_size,
      "| smallest", merged_smallest_group, merged_smallest_size)
print("Split sizes:", len(train_ids), "train /", len(val_ids), "validation")
print("Q1 ANSWER FORMAT:", f"{clean_count}; {positive_by_class['defect_a']}, {positive_by_class['defect_b']}, {positive_by_class['defect_c']}, {positive_by_class['defect_d']}; {q1_unique_combinations}; {q1_largest_size}, {q1_smallest_size}")

split_path = ARTIFACT_DIR / "milestone4_seed42_split.json"
with open(split_path, "w") as f:
    json.dump({"train_ids": train_ids, "validation_ids": val_ids,
               "working_ids": WORKING_IDS, "seed": 42}, f, indent=2)
print("Saved reproducible split:", split_path)

Q1(a) Completely clean images: 578
Q1(b) Positive images [defect_a, defect_b, defect_c, defect_d]: [107, 35, 438, 181]
Q1(c) Unique raw combination strings (before merging): 15
Q1(d) Largest raw group: clean 578
Q1(d) Smallest raw group: defect_a+defect_b+defect_c 1
Rare groups merged into rare_other: ['defect_a+defect_b', 'defect_a+defect_b+defect_c', 'defect_b+defect_c', 'defect_b+defect_c+defect_d']
Split groups after merging:
 {'clean': 578, 'defect_c': 334, 'defect_d': 81, 'defect_c+defect_d': 71, 'defect_a': 59, 'defect_b': 22, 'defect_a+defect_c': 21, 'defect_a+defect_d': 14, 'rare_other': 7, 'defect_a+defect_c+defect_d': 7, 'defect_a+defect_b+defect_d': 3, 'defect_b+defect_d': 3}
Merged-group extrema (additional diagnostic): largest clean 578 | smallest defect_a+defect_b+defect_d 3
Split sizes: 960 train / 240 validation
Q1 ANSWER FORMAT: 578; 107, 35, 438, 181; 15; 578, 1
Saved reproducible split: /content/drive/MyDrive/DLGenAI_Milestone4/milestone4_seed42_split.json


## 6. Q2 — Pixel-level imbalance and defect_b `pos_weight`

In [7]:
b_idx = CLASSES.index("defect_b")
b_areas = AREA[:, b_idx].astype(np.int64)
b_positive = b_areas > 0
b_total_positive_pixels = int(b_areas.sum())
b_total_pixels = 1200 * N_PIXELS
b_positive_percentage = 100.0 * b_total_positive_pixels / b_total_pixels
b_mean_positive_area = float(b_areas[b_positive].mean()) if b_positive.any() else 0.0
b_negative_pixels = b_total_pixels - b_total_positive_pixels
b_pos_weight_ratio = b_negative_pixels / max(b_total_positive_pixels, 1)
b_pos_weight_rounded = int(math.floor(b_pos_weight_ratio + 0.5))

print(f"Q2(a) Positive defect_b pixels: {b_positive_percentage:.4f}% of all 1200×102,400 pixels")
print(f"Q2(b) Mean defect_b area among positive images: {b_mean_positive_area:.1f} pixels")
print(f"Q2(c) Negative:positive pixel ratio / candidate BCE pos_weight: {b_pos_weight_rounded}")
print("Q2 ANSWER FORMAT:", f"{b_positive_percentage:.4f}%; {b_mean_positive_area:.1f}; {b_pos_weight_rounded}")

Q2(a) Positive defect_b pixels: 0.0160% of all 1200×102,400 pixels
Q2(b) Mean defect_b area among positive images: 561.5 pixels
Q2(c) Negative:positive pixel ratio / candidate BCE pos_weight: 6251
Q2 ANSWER FORMAT: 0.0160%; 561.5; 6251


## 7. Q3 — Bounded square-root sampling (all 1,200 images; no rare-group merging)

In [8]:
# For each raw combination group g: w_g = min(sqrt(n_clean/n_g), 5).
n_clean = int(raw_group_sizes.get("clean", 0))
if n_clean <= 0:
    raise RuntimeError("The 1,200-image subset has no clean images; Q3 formula needs n_clean > 0.")
group_weights = {g: min(math.sqrt(n_clean / int(n_g)), 5.0)
                 for g, n_g in raw_group_sizes.items()}
group_mass = {g: int(raw_group_sizes[g]) * group_weights[g] for g in raw_group_sizes.index}
total_mass = float(sum(group_mass.values()))
sampler_group_prob = {g: mass / total_mass for g, mass in group_mass.items()}
clean_sampler_fraction = sampler_group_prob["clean"]
clean_natural_fraction = n_clean / 1200.0
b_positive_groups = [g for g in raw_group_sizes.index if "defect_b" in g.split("+")]
b_images = int(PRESENCE[:, b_idx].sum())
b_sampler_expected = 1200.0 * sum(sampler_group_prob[g] for g in b_positive_groups)
b_natural_expected = float(b_images)
b_enrichment = b_sampler_expected / max(b_natural_expected, 1e-12)
cap_group_count = sum(1 for value in group_weights.values() if math.isclose(value, 5.0))

print(f"Q3(a) Clean draws — bounded sampler: {100*clean_sampler_fraction:.2f}%; natural: {100*clean_natural_fraction:.2f}%")
print(f"Q3(b) defect_b-positive draws/epoch — sampler: {b_sampler_expected:.2f}; natural: {b_natural_expected:.2f}")
print(f"Q3(c) defect_b-positive enrichment factor: {b_enrichment:.2f}x")
print(f"Q3(d) Combination groups at weight cap 5: {cap_group_count}")
print("Q3 ANSWER FORMAT:", f"{100*clean_sampler_fraction:.2f}%, {100*clean_natural_fraction:.2f}%; {b_sampler_expected:.2f}, {b_natural_expected:.2f}; {b_enrichment:.2f}x; {cap_group_count}")

group_weight_table = pd.DataFrame({
    "group_size": raw_group_sizes,
    "sampler_weight": pd.Series(group_weights),
    "per_image_sampling_mass": pd.Series(group_mass),
    "expected_draws_per_epoch": pd.Series({g: 1200*sampler_group_prob[g] for g in sampler_group_prob}),
}).sort_values("group_size", ascending=False)
display(group_weight_table.round(4))

Q3(a) Clean draws — bounded sampler: 28.81%; natural: 48.17%
Q3(b) defect_b-positive draws/epoch — sampler: 104.69; natural: 35.00
Q3(c) defect_b-positive enrichment factor: 2.99x
Q3(d) Combination groups at weight cap 5: 10
Q3 ANSWER FORMAT: 28.81%, 48.17%; 104.69, 35.00; 2.99x; 10


,group_size,sampler_weight,per_image_sampling_mass,expected_draws_per_epoch
clean,578,1.0000,578.0000,345.7632
defect_c,334,1.3155,439.3768,262.8380
defect_d,81,2.6713,216.3747,129.4367
defect_c+defect_d,71,2.8532,202.5784,121.1836
defect_a,59,3.1300,184.6673,110.4691
defect_b,22,5.0000,110.0000,65.8027
defect_a+defect_c,21,5.0000,105.0000,62.8117
defect_a+defect_d,14,5.0000,70.0000,41.8744
defect_a+defect_c+defect_d,7,5.0000,35.0000,20.9372
defect_b+defect_d,3,5.0000,15.0000,8.9731


## 8. Q4–Q6 — Exact analytical answers

In [9]:
# Q4: base LR = 3e-4; 60-step warmup, then a 420-step cosine decay.
BASE_LR_ANALYTIC = 3e-4

def q4_multiplier(t):
    if t < 60:
        return t / 60.0
    return 0.5 * (1.0 + math.cos(math.pi * (t - 60) / 420.0))

q4_lrs = {t: BASE_LR_ANALYTIC * q4_multiplier(t) for t in [15, 270, 400]}
q4_bug_lr = BASE_LR_ANALYTIC * q4_multiplier(8)  # scheduler.step() called once per epoch
print("Q4(a) LR at steps 15, 270, 400:", [f"{q4_lrs[t]:.2e}" for t in [15,270,400]])
print(f"Q4(b) Buggy LR after epoch 8 (scheduler advanced 8 steps): {q4_bug_lr:.2e}")
print("Q4(c) Symptom: the warmup is stretched from 60 optimizer steps to roughly 60 epochs, so the LR stays far too small and the loss/score curves learn unusually slowly or appear stalled.")
print("Q4 ANSWER FORMAT:", f"{q4_lrs[15]:.2e}, {q4_lrs[270]:.2e}, {q4_lrs[400]:.2e}; {q4_bug_lr:.2e}; scheduler.step() is called once per epoch instead of after every optimizer step, stretching the warmup and slowing learning.")

# Q5: strict condition score > best + min_delta.
trace = [0.512,0.601,0.648,0.661,0.664,0.668,0.671,0.670,0.672,0.669,0.673,0.665]
def simulate_early_stopping(scores, patience=3, min_delta=0.002):
    best = -float("inf"); best_epoch = None; bad = 0; stop_epoch = None
    for epoch, score in enumerate(scores, start=1):
        if score > best + min_delta:
            best, best_epoch, bad = score, epoch, 0
        else:
            bad += 1
            if bad >= patience:
                stop_epoch = epoch
                break
    return stop_epoch, best_epoch, best
q5_stop, q5_restore_epoch, q5_restore_score = simulate_early_stopping(trace, 3, 0.002)
q5_zero_stop, q5_zero_epoch, q5_zero_score = simulate_early_stopping(trace, 3, 0.0)
print(f"Q5(a) Stop epoch: {q5_stop}; restore epoch: {q5_restore_epoch}; restored score: {q5_restore_score:.3f}")
print(f"Q5(b) min_delta=0: stop within 12 epochs? {'yes' if q5_zero_stop else 'no'}; best epoch: {q5_zero_epoch}; best score: {q5_zero_score:.3f}")
print("Q5 ANSWER FORMAT:", f"{q5_stop}, {q5_restore_epoch}, {q5_restore_score:.3f}; {'yes' if q5_zero_stop else 'no'}, {q5_zero_score:.3f}")

# Q6: decimal MB, 25M parameters, FP32 weights+gradients+two AdamW moments.
parameter_count = 25_000_000
state_mb = parameter_count * 4 * 4 / 1_000_000
fp32_batch16_mb = state_mb + 16 * 150
amp_batch16_mb = state_mb + 16 * 75
max_batch_fp32 = int((3400 - state_mb) // 150)
max_batch_amp = int((3400 - state_mb) // 75)
train_images = 960
micro_batch = 8
accumulation = 4
micro_batches_per_epoch = math.ceil(train_images / micro_batch)
optimizer_steps_per_epoch = math.ceil(micro_batches_per_epoch / accumulation)
effective_batch = micro_batch * accumulation
warmup_steps_2_epochs = 2 * optimizer_steps_per_epoch
print(f"Q6(a) Weights + gradients + two AdamW moments: {state_mb:.0f} MB")
print(f"Q6(b) Total batch-16 memory — FP32: {fp32_batch16_mb:.0f} MB; AMP: {amp_batch16_mb:.0f} MB")
print(f"Q6(c) Largest batch under 3,400 MB — FP32: {max_batch_fp32}; AMP: {max_batch_amp}")
print(f"Q6(d) Effective batch size: {effective_batch}")
print(f"Q6(e) Micro-batches/epoch: {micro_batches_per_epoch}; optimizer steps/epoch: {optimizer_steps_per_epoch}")
print(f"Q6(f) Divide each micro-batch loss by: {accumulation}")
print(f"Q6(g) 2-epoch warmup in optimizer steps: {warmup_steps_2_epochs}")
print("Q6 ANSWER FORMAT:", f"{state_mb:.0f} MB; {fp32_batch16_mb:.0f}, {amp_batch16_mb:.0f} MB; {max_batch_fp32}, {max_batch_amp}; {effective_batch}; {micro_batches_per_epoch}, {optimizer_steps_per_epoch}; {accumulation}; {warmup_steps_2_epochs}")

Q4(a) LR at steps 15, 270, 400: ['7.50e-05', '1.50e-04', '2.61e-05']
Q4(b) Buggy LR after epoch 8 (scheduler advanced 8 steps): 4.00e-05
Q4(c) Symptom: the warmup is stretched from 60 optimizer steps to roughly 60 epochs, so the LR stays far too small and the loss/score curves learn unusually slowly or appear stalled.
Q4 ANSWER FORMAT: 7.50e-05, 1.50e-04, 2.61e-05; 4.00e-05; scheduler.step() is called once per epoch instead of after every optimizer step, stretching the warmup and slowing learning.
Q5(a) Stop epoch: 10; restore epoch: 7; restored score: 0.671
Q5(b) min_delta=0: stop within 12 epochs? no; best epoch: 11; best score: 0.673
Q5 ANSWER FORMAT: 10, 7, 0.671; no, 0.673
Q6(a) Weights + gradients + two AdamW moments: 400 MB
Q6(b) Total batch-16 memory — FP32: 2800 MB; AMP: 1600 MB
Q6(c) Largest batch under 3,400 MB — FP32: 20; AMP: 40
Q6(d) Effective batch size: 32
Q6(e) Micro-batches/epoch: 120; optimizer steps/epoch: 30
Q6(f) Divide each micro-batch loss by: 4
Q6(g) 2-epoch wa

## 9. Q7 — Naive Dice trap versus the official metric

In [10]:
total_pairs = 1200 * N_CLASSES
positive_pairs = int(PRESENCE.sum())
empty_pairs = total_pairs - positive_pairs
baseline_naive_dice = empty_pairs / total_pairs  # all-empty prediction: empty-empty=1, positive-empty=0
baseline_foreground_dice = 0.0
baseline_fp_score = 1.0
baseline_empty_accuracy = 1.0
baseline_official_score = 0.70*baseline_foreground_dice + 0.20*baseline_fp_score + 0.10*baseline_empty_accuracy

# Q7 toy model: 60 positive pairs with mean foreground Dice=.50; 340 empty pairs,
# predicts empty on 306, and has a 1,024-pixel false blob on each remaining 34.
toy_positive_pairs, toy_foreground_dice = 60, 0.50
toy_empty_pairs, toy_correct_empty = 340, 306
toy_false_blob_pairs, toy_blob_pixels = 34, 1024
toy_fp_pixels = toy_false_blob_pairs * toy_blob_pixels
toy_fp_score = 1.0 - toy_fp_pixels / (toy_empty_pairs * N_PIXELS)
toy_empty_accuracy = toy_correct_empty / toy_empty_pairs
toy_official_score = 0.70*toy_foreground_dice + 0.20*toy_fp_score + 0.10*toy_empty_accuracy
toy_naive_dice = (toy_positive_pairs*toy_foreground_dice + toy_correct_empty) / 400.0
naive_metric_verdict = (
    "empty-aware mean Dice ranks the toy model below the all-empty baseline"
    if baseline_naive_dice > toy_naive_dice else
    "empty-aware mean Dice does not rank the toy model below the all-empty baseline on this subset"
)
print(f"Q7(a) Empty ground-truth image-class pairs: {empty_pairs}")
print(f"Q7(b) All-empty baseline naive empty-aware mean Dice: {baseline_naive_dice:.4f}")
print(f"Q7(c) All-empty baseline official score: {baseline_official_score:.4f}")
print(f"Q7(d) Toy model official score: {toy_official_score:.4f}; toy naive Dice: {toy_naive_dice:.4f}")
print(f"Q7(e) {naive_metric_verdict}; baseline naive Dice = {baseline_naive_dice:.4f}.")
print("Q7 ANSWER FORMAT (Part A):", f"{empty_pairs}; {baseline_naive_dice:.4f}; {baseline_official_score:.4f}")
print("Q7 ANSWER FORMAT (Part B):", f"{toy_official_score:.4f}, {toy_naive_dice:.4f}; {naive_metric_verdict}; {baseline_naive_dice:.4f}")

Q7(a) Empty ground-truth image-class pairs: 4039
Q7(b) All-empty baseline naive empty-aware mean Dice: 0.8415
Q7(c) All-empty baseline official score: 0.3000
Q7(d) Toy model official score: 0.6398; toy naive Dice: 0.8400
Q7(e) empty-aware mean Dice ranks the toy model below the all-empty baseline; baseline naive Dice = 0.8415.
Q7 ANSWER FORMAT (Part A): 4039; 0.8415; 0.3000
Q7 ANSWER FORMAT (Part B): 0.6398, 0.8400; empty-aware mean Dice ranks the toy model below the all-empty baseline; 0.8415


## 10. Q9 — Defect-size bins and missed-pair analysis

In [11]:
# Pool all positive image-class pairs from all four classes.
positive_areas = AREA[AREA > 0].astype(np.int64)
bin_masks = {
    "1-100": (positive_areas >= 1) & (positive_areas <= 100),
    "101-500": (positive_areas >= 101) & (positive_areas <= 500),
    "501+": positive_areas >= 501,
}
bin_counts = {name: int(mask.sum()) for name, mask in bin_masks.items()}
total_positive_pixels = int(positive_areas.sum())
large_bin_pixels = int(positive_areas[bin_masks["501+"]].sum())
large_pixel_percentage = 100.0 * large_bin_pixels / max(total_positive_pixels, 1)
recall_by_bin = {"1-100": 0.30, "101-500": 0.70, "501+": 0.95}
missed_expected = {name: bin_counts[name] * (1.0 - recall_by_bin[name]) for name in bin_counts}
missed_total = float(sum(missed_expected.values()))
most_missed_bin = max(missed_expected, key=missed_expected.get)
print("Q9(a) Positive pairs by bin:", bin_counts)
print(f"Q9(b) Share of all positive GT pixels in 501+ bin: {large_pixel_percentage:.2f}%")
print(f"Q9(c) Expected missed positive pairs: {missed_total:.2f}; largest contributor: {most_missed_bin} ({missed_expected[most_missed_bin]:.2f})")
print("Q9(d) The bin with the most missed pairs is driven by how many pairs it contains and its miss rate, whereas missed pixels also depend on the defect area inside each pair.")
print("Q9 ANSWER FORMAT:", f"{bin_counts['1-100']}, {bin_counts['101-500']}, {bin_counts['501+']}; {large_pixel_percentage:.2f}%; {missed_total:.2f}, {most_missed_bin}; pair counts and pixel areas measure different quantities.")

Q9(a) Positive pairs by bin: {'1-100': 5, '101-500': 99, '501+': 657}
Q9(b) Share of all positive GT pixels in 501+ bin: 99.24%
Q9(c) Expected missed positive pairs: 66.05; largest contributor: 501+ (32.85)
Q9(d) The bin with the most missed pairs is driven by how many pairs it contains and its miss rate, whereas missed pixels also depend on the defect area inside each pair.
Q9 ANSWER FORMAT: 5, 99, 657; 99.24%; 66.05, 501+; pair counts and pixel areas measure different quantities.


## 11. Prepare the 960/240 training data and baseline model

**Architecture default:** U-Net with an ImageNet-pretrained ResNet34 encoder, consistent with the project's established segmentation baseline. If your M2/M3 baseline architecture or loss differs, change the configuration below to match it exactly before running Q8/Q10.

In [12]:
# BASELINE MODEL CONFIG — align this with your exact Milestone 2/3 baseline if needed.
MODEL_NAME = "Unet"
ENCODER_NAME = "resnet34"
ENCODER_WEIGHTS = "imagenet"
LOSS_DESCRIPTION = "0.5 * BCEWithLogits + 0.5 * positive-target soft Dice"
AUGMENT_FLIPS = False  # Keep False unless your exact M2/M3 baseline used these flips.

id_to_mask = {image_id: MASKS[i] for i, image_id in enumerate(WORKING_IDS)}

class SurfaceGuardM4Dataset(Dataset):
    def __init__(self, image_ids, augment=False, seed=42):
        self.image_ids = list(map(str, image_ids))
        self.augment = augment
        self.seed = seed
    def __len__(self):
        return len(self.image_ids)
    def __getitem__(self, idx):
        image_id = self.image_ids[idx]
        path = TRAIN_IMAGE_DIR / f"{image_id}.jpg"
        if not path.is_file():
            raise FileNotFoundError(path)
        with Image.open(path) as im:
            arr = np.asarray(im.convert("L"), dtype=np.float32) / 255.0
        if arr.shape != (HEIGHT, WIDTH):
            arr = np.asarray(Image.fromarray((arr*255).astype(np.uint8)).resize((WIDTH, HEIGHT), Image.Resampling.BILINEAR), dtype=np.float32)/255.0
        mean, std = float(arr.mean()), max(float(arr.std()), 0.05)
        arr = np.ascontiguousarray((arr-mean)/std, dtype=np.float32)
        mask = id_to_mask[image_id].astype(np.float32, copy=True)
        x = torch.from_numpy(arr).unsqueeze(0)
        y = torch.from_numpy(mask)
        if self.augment:
            if random.random() < 0.5:
                x = torch.flip(x, dims=[2]); y = torch.flip(y, dims=[2])
            if random.random() < 0.5:
                x = torch.flip(x, dims=[1]); y = torch.flip(y, dims=[1])
        return x, y

train_dataset = SurfaceGuardM4Dataset(train_ids, augment=AUGMENT_FLIPS)
val_dataset = SurfaceGuardM4Dataset(val_ids, augment=False)


def make_baseline_model():
    if MODEL_NAME == "Unet":
        return smp.Unet(
            encoder_name=ENCODER_NAME, encoder_weights=ENCODER_WEIGHTS,
            in_channels=1, classes=N_CLASSES, activation=None
        )
    raise NotImplementedError("Set make_baseline_model() to the exact architecture used in your M2/M3 baseline.")


def segmentation_loss(logits, targets):
    logits = logits.float(); targets = targets.float()
    bce = F.binary_cross_entropy_with_logits(logits, targets)
    probs = torch.sigmoid(logits)
    # Positive-target soft Dice: average Dice only over image/class pairs with GT foreground.
    dims = (2, 3)
    intersection = (probs * targets).sum(dim=dims)
    denominator = probs.sum(dim=dims) + targets.sum(dim=dims)
    pair_dice = (2.0 * intersection + 1.0) / (denominator + 1.0)
    positive_pairs = targets.sum(dim=dims) > 0
    if positive_pairs.any():
        dice_loss = 1.0 - pair_dice[positive_pairs].mean()
    else:
        dice_loss = probs.mean() * 0.0
    return 0.5*bce + 0.5*dice_loss

print("Training split:", len(train_dataset), "| validation split:", len(val_dataset))
print("Model:", MODEL_NAME, ENCODER_NAME, "| encoder weights:", ENCODER_WEIGHTS)
print("Loss:", LOSS_DESCRIPTION, "| flip augmentation:", AUGMENT_FLIPS)
print("Batch size:", BATCH_SIZE, "| AMP:", AMP_ENABLED, "| epochs:", EPOCHS)

Training split: 960 | validation split: 240
Model: Unet resnet34 | encoder weights: imagenet
Loss: 0.5 * BCEWithLogits + 0.5 * positive-target soft Dice | flip augmentation: False
Batch size: 16 | AMP: True | epochs: 6


## 12. W&B setup — required by the milestone

In [13]:
import wandb

WANDB_HOST = "https://forge.coreweave.com/wandb"
WANDB_ENTITY = "23f3003529-dl-genai-project"
WANDB_PROJECT = "23f3003529-t32026"

try:
    WANDB_API_KEY = userdata.get("WANDB_API_KEY")
except Exception:
    WANDB_API_KEY = None
if not WANDB_API_KEY:
    WANDB_API_KEY = getpass.getpass("Enter your W&B Forge API key (input hidden): ")
if not WANDB_API_KEY or not WANDB_API_KEY.strip():
    raise RuntimeError("W&B API key is required because Milestone 4 requires every training run to be logged.")

wandb.login(key=WANDB_API_KEY.strip(), host=WANDB_HOST, relogin=True)
del WANDB_API_KEY
print("W&B authenticated. Entity/project:", WANDB_ENTITY, "/", WANDB_PROJECT)

/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: Updated settings file /root/.config/wandb/settings
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://forge.coreweave.com/wandb.
wandb: No netrc file found, creating one.
wandb: Appending key for forge.coreweave.com to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


W&B authenticated. Entity/project: 23f3003529-dl-genai-project / 23f3003529-t32026


## 13. Training utilities, official metric and schedulers

In [14]:
from torch.cuda.amp import GradScaler


def warmup_cosine_multiplier(step_1based):
    """PDF Q4 schedule: 60 warmup optimizer steps + 420 cosine steps (480 total)."""
    t = int(step_1based)
    if t < 60:
        return t / 60.0
    return 0.5 * (1.0 + math.cos(math.pi * (t - 60) / 420.0))


def calculate_official_metric(targets_np, probs_np, threshold=0.5):
    """Pooled official 0.70*foreground Dice + 0.20*FP score + 0.10*empty accuracy."""
    targets = np.asarray(targets_np) >= 0.5
    predictions = np.asarray(probs_np) >= threshold
    target_counts = targets.sum(axis=(2,3))
    pred_counts = predictions.sum(axis=(2,3))
    intersections = np.logical_and(targets, predictions).sum(axis=(2,3))
    positive = target_counts > 0
    empty = ~positive
    positive_pairs = int(positive.sum())
    empty_pairs = int(empty.sum())
    if positive_pairs == 0 or empty_pairs == 0:
        raise ValueError("Validation metric requires both positive and empty image-class pairs.")
    pair_dice = 2.0*intersections[positive] / np.maximum(pred_counts[positive] + target_counts[positive], 1)
    foreground_dice = float(pair_dice.mean())
    fp_pixels = int(pred_counts[empty].sum())
    fp_score = 1.0 - fp_pixels / (empty_pairs * N_PIXELS)
    empty_accuracy = float((pred_counts[empty] == 0).mean())
    official = 0.70*foreground_dice + 0.20*fp_score + 0.10*empty_accuracy
    false_alarm_rate = float((pred_counts[empty] > 0).mean() * 100.0)
    return {
        "official_score": float(official),
        "foreground_dice": foreground_dice,
        "false_positive_score": float(fp_score),
        "empty_accuracy": empty_accuracy,
        "false_alarm_rate_empty_pairs_pct": false_alarm_rate,
        "positive_pairs": positive_pairs,
        "empty_pairs": empty_pairs,
    }


@torch.no_grad()
def evaluate_model(model, loader):
    model.eval()
    loss_total, samples = 0.0, 0
    target_batches, prob_batches = [], []
    for images, masks in tqdm(loader, leave=False, desc="Validation"):
        images = images.to(DEVICE, non_blocking=(DEVICE.type=="cuda"))
        masks = masks.to(DEVICE, non_blocking=(DEVICE.type=="cuda"))
        with (torch.autocast("cuda", dtype=torch.float16) if AMP_ENABLED else nullcontext()):
            logits = model(images)
        loss = segmentation_loss(logits.float(), masks.float())
        probs = torch.sigmoid(logits.float())
        loss_total += float(loss.item()) * len(images)
        samples += len(images)
        target_batches.append(masks.cpu().numpy().astype(np.uint8))
        prob_batches.append(probs.cpu().numpy().astype(np.float32))
    targets_np = np.concatenate(target_batches, axis=0)
    probs_np = np.concatenate(prob_batches, axis=0)
    metric = calculate_official_metric(targets_np, probs_np, DECISION_THRESHOLD)
    metric["val_loss"] = loss_total / max(samples,1)
    return metric


def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed); random.seed(worker_seed)


def make_train_loader(seed, sampling="natural"):
    generator = torch.Generator()
    generator.manual_seed(seed)
    if sampling == "natural":
        return DataLoader(
            train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=False,
            num_workers=2 if DEVICE.type=="cuda" else 0,
            pin_memory=(DEVICE.type=="cuda"), generator=generator, worker_init_fn=seed_worker
        )
    if sampling == "bounded_sqrt":
        combo_by_id = dict(zip(WORKING_IDS, COMBINATIONS))
        train_combo = {image_id: str(combo_by_id[image_id]) for image_id in train_ids}
        counts = pd.Series(list(train_combo.values())).value_counts()
        n_clean_train = int(counts.get("clean",0))
        if n_clean_train <= 0:
            raise RuntimeError("Training split has no clean combination group; sampler formula is undefined.")
        weights_by_group = {g: min(math.sqrt(n_clean_train/int(n)), 5.0) for g,n in counts.items()}
        per_sample = torch.as_tensor([weights_by_group[train_combo[i]] for i in train_ids], dtype=torch.double)
        sampler = WeightedRandomSampler(per_sample, num_samples=len(train_dataset), replacement=True, generator=generator)
        return DataLoader(
            train_dataset, batch_size=BATCH_SIZE, sampler=sampler, drop_last=False,
            num_workers=2 if DEVICE.type=="cuda" else 0,
            pin_memory=(DEVICE.type=="cuda"), worker_init_fn=seed_worker
        )
    raise ValueError(f"Unknown sampling mode: {sampling}")


def run_training_experiment(run_name, seed=42, schedule="constant", sampling="natural"):
    """Train for six fixed epochs; best checkpoint is selected by official validation score."""
    set_seed(seed)
    model = make_baseline_model().to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=BASE_LR)
    scaler = GradScaler(enabled=AMP_ENABLED)
    loader = make_train_loader(seed, sampling=sampling)
    steps_per_epoch = len(loader)
    expected_steps = 60
    if steps_per_epoch != expected_steps:
        raise RuntimeError(f"Expected 60 optimizer steps/epoch, got {steps_per_epoch}.")

    run_dir = ARTIFACT_DIR / run_name
    run_dir.mkdir(parents=True, exist_ok=True)
    best_path = run_dir / "best_checkpoint.pt"
    history = []
    best_score = -float("inf")
    best_epoch = None
    best_metrics = None
    global_step = 0

    wb = wandb.init(
        entity=WANDB_ENTITY, project=WANDB_PROJECT, name=f"M4-{run_name}",
        job_type="milestone4-training",
        tags=["DLGenAI-M4", run_name, schedule, sampling],
        config={
            "run_name":run_name, "seed":seed, "epochs":EPOCHS,
            "image_size":[HEIGHT,WIDTH], "model":MODEL_NAME,
            "encoder":ENCODER_NAME, "encoder_weights":ENCODER_WEIGHTS,
            "optimizer":"AdamW", "base_lr":BASE_LR, "batch_size":BATCH_SIZE,
            "amp_requested":True, "amp_enabled":AMP_ENABLED,
            "scheduler":schedule, "sampling":sampling,
            "threshold":DECISION_THRESHOLD, "loss":LOSS_DESCRIPTION,
            "train_images":len(train_ids), "validation_images":len(val_ids),
        }
    )

    try:
        for epoch in range(1, EPOCHS+1):
            model.train()
            running_loss, seen = 0.0, 0
            lr_at_epoch_start = float(optimizer.param_groups[0]["lr"])
            for images, masks in tqdm(loader, desc=f"{run_name} | epoch {epoch}/{EPOCHS}"):
                images = images.to(DEVICE, non_blocking=(DEVICE.type=="cuda"))
                masks = masks.to(DEVICE, non_blocking=(DEVICE.type=="cuda"))
                if schedule == "warmup_cosine":
                    lr_now = BASE_LR * warmup_cosine_multiplier(global_step + 1)
                    for group in optimizer.param_groups:
                        group["lr"] = lr_now
                else:
                    lr_now = BASE_LR
                    for group in optimizer.param_groups:
                        group["lr"] = BASE_LR

                optimizer.zero_grad(set_to_none=True)
                with (torch.autocast("cuda", dtype=torch.float16) if AMP_ENABLED else nullcontext()):
                    logits = model(images)
                    loss = segmentation_loss(logits, masks)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
                running_loss += float(loss.item()) * len(images)
                seen += len(images)
                global_step += 1

            train_loss = running_loss / max(seen,1)
            val_metrics = evaluate_model(model, DataLoader(
                val_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=False,
                num_workers=2 if DEVICE.type=="cuda" else 0,
                pin_memory=(DEVICE.type=="cuda")
            ))
            row = {
                "epoch":epoch, "train_loss":float(train_loss),
                "val_loss":float(val_metrics["val_loss"]),
                "official_val_score":float(val_metrics["official_score"]),
                "foreground_dice":float(val_metrics["foreground_dice"]),
                "false_positive_score":float(val_metrics["false_positive_score"]),
                "empty_accuracy":float(val_metrics["empty_accuracy"]),
                "false_alarm_rate_empty_pairs_pct":float(val_metrics["false_alarm_rate_empty_pairs_pct"]),
                "lr_last_step":float(lr_now), "lr_epoch_start":float(lr_at_epoch_start),
            }
            history.append(row)
            wandb.log(row, step=epoch)
            print(
                f"{run_name} | epoch {epoch}: train_loss={train_loss:.5f}, "
                f"val_loss={val_metrics['val_loss']:.5f}, official={val_metrics['official_score']:.5f}, "
                f"lr={lr_now:.2e}"
            )

            if val_metrics["official_score"] > best_score:
                best_score = float(val_metrics["official_score"])
                best_epoch = epoch
                best_metrics = dict(val_metrics)
                torch.save({
                    "model_state_dict": model.state_dict(),
                    "epoch": epoch, "official_val_score": best_score,
                    "classes": CLASSES, "height": HEIGHT, "width": WIDTH,
                    "run_name": run_name, "seed": seed,
                    "schedule": schedule, "sampling": sampling,
                }, best_path)

        history_df = pd.DataFrame(history)
        history_path = run_dir / "history.csv"
        history_df.to_csv(history_path, index=False)
        summary = {
            "run_name":run_name, "seed":seed, "schedule":schedule, "sampling":sampling,
            "best_epoch_by_official_score":int(best_epoch),
            "best_official_val_score":float(best_score),
            "best_metrics":best_metrics,
            "checkpoint":str(best_path), "history_csv":str(history_path),
            "epochs_completed":len(history),
        }
        with open(run_dir / "summary.json", "w") as f:
            json.dump(summary, f, indent=2)
        wb.summary.update({
            "best_epoch_by_official_score":int(best_epoch),
            "best_official_val_score":float(best_score),
            "best_val_loss":float(history_df["val_loss"].min()),
            "checkpoint_path":str(best_path),
        })
        wb.finish()
        return {"summary":summary, "history":history_df}
    except Exception:
        wb.finish(exit_code=1)
        raise
    finally:
        del model, optimizer, scaler, loader
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

print("Training utilities ready.")

Training utilities ready.


In [15]:

import wandb
from pathlib import Path

ORIGINAL_WANDB_INIT = wandb.init

WANDB_RUN_DIR = ARTIFACT_DIR / "wandb_runs"
WANDB_RUN_DIR.mkdir(parents=True, exist_ok=True)

def resilient_wandb_init(*args, **kwargs):
    kwargs = dict(kwargs)
    kwargs.setdefault("dir", str(WANDB_RUN_DIR))

    # First try normal online tracking with a longer timeout.
    try:
        print("Attempting W&B online initialisation...")
        return ORIGINAL_WANDB_INIT(
            *args,
            **kwargs,
            settings=wandb.Settings(init_timeout=180)
        )

    except Exception as online_error:
        print("Online W&B initialisation failed:")
        print(type(online_error).__name__, str(online_error)[:200])

        try:
            wandb.finish(quiet=True)
        except Exception:
            pass

    # Retry in offline mode. Metrics will be stored locally.
    print("Switching to W&B offline mode...")

    offline_kwargs = dict(kwargs)
    offline_kwargs["mode"] = "offline"
    offline_kwargs["reinit"] = True

    return ORIGINAL_WANDB_INIT(
        *args,
        **offline_kwargs,
        settings=wandb.Settings(init_timeout=60)
    )

# Patch the existing training function without redefining it.
wandb.init = resilient_wandb_init

print("W&B timeout fix installed.")
print("Next: rerun the Q8 training cell.")


W&B timeout fix installed.
Next: rerun the Q8 training cell.


## 14. Q8 — Train the six-epoch baseline with warmup + cosine; log every epoch to W&B

In [16]:
# Q8 explicitly requests the Q4 warmup+cosine schedule.
q8_result = run_training_experiment(
    run_name="Q8_baseline_warmup_cosine_seed42",
    seed=42, schedule="warmup_cosine", sampling="natural"
)
q8_history = q8_result["history"]
q8_summary = q8_result["summary"]
q8_lowest_val_epoch = int(q8_history.loc[q8_history["val_loss"].idxmin(), "epoch"])
q8_best_score_epoch = int(q8_summary["best_epoch_by_official_score"])
q8_best_score = float(q8_summary["best_official_val_score"])
q8_gap_epoch6 = float(q8_history.iloc[-1]["val_loss"] - q8_history.iloc[-1]["train_loss"])

# Simple transparent learning-curve description based on actual six-epoch trend.
train_drop = float(q8_history["train_loss"].iloc[0] - q8_history["train_loss"].iloc[-1])
val_drop = float(q8_history["val_loss"].iloc[0] - q8_history["val_loss"].iloc[-1])
score_trend = float(q8_history["official_val_score"].iloc[-1] - q8_history["official_val_score"].iloc[0])
if train_drop > 0 and val_drop > 0 and score_trend > 0:
    q8_curve_pattern = "healthy convergence over the observed six epochs"
elif train_drop > 0 and val_drop < 0 and score_trend <= 0:
    q8_curve_pattern = "overfitting: train loss falls while validation worsens"
else:
    q8_curve_pattern = "mixed/early learning trend; inspect all six epoch curves"

print("Q8 epoch history:")
display(q8_history.round(6))
print(f"Q8(a) Lowest validation loss epoch: {q8_lowest_val_epoch}")
print(f"Q8(b) Highest official validation score: epoch {q8_best_score_epoch}, {q8_best_score:.3f}")
print(f"Q8(c) Generalization gap at epoch 6 (val loss - train loss): {q8_gap_epoch6:.3f}")
print(f"Q8(d) Monitor official validation score; keep epoch {q8_best_score_epoch}; curve: {q8_curve_pattern}.")
print("Q8 ANSWER FORMAT:", f"{q8_lowest_val_epoch}; {q8_best_score_epoch}, {q8_best_score:.3f}; {q8_gap_epoch6:.3f}; monitor official validation score and keep epoch {q8_best_score_epoch} — {q8_curve_pattern}.")

config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 87.3MB            

model.safetensors: downloading bytes:           |  0.00B            

/tmp/ipykernel_354/2451287887.py:103: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=AMP_ENABLED)


Attempting W&B online initialisation...


Online W&B initialisation failed:
CommError Run initialization has timed out after 180.0 sec. Please try increasing the timeout with the `init_timeout` setting: `wandb.init(settings=wandb.Settings(init_timeout=120))`.
Switching to W&B offline mode...


... (17 outputs left out)

Validation:   0%|          | 0/15 [00:00<?, ?it/s]

Q8_baseline_warmup_cosine_seed42 | epoch 6: train_loss=0.35810, val_loss=0.37164, official=0.56892, lr=5.65e-05


empty_accuracy,▁▂▇▇▇█
epoch,▁▂▄▅▇█
false_alarm_rate_empty_pairs_pct,█▇▂▂▂▁
false_positive_score,▁██▇██
foreground_dice,▁▄▆▇▇█
lr_epoch_start,██▇▆▄▁
lr_last_step,██▆▅▃▁
official_val_score,▁▄▆▇▇█
train_loss,█▅▄▂▂▁
val_loss,█▅▃▃▂▁
best_epoch_by_official_score,6


NameError: name 'gc' is not defined

In [17]:
# Q8 RECOVERY — DOES NOT RUN TRAINING OR REQUIRE A GPU

import gc
import json
import pandas as pd
from pathlib import Path

# Fix the missing import for any later cleanup calls.
gc.collect()

# Locate the files saved by the completed Q8 run.
q8_run_dir = ARTIFACT_DIR / "Q8_baseline_warmup_cosine_seed42"

history_path = q8_run_dir / "history.csv"
summary_path = q8_run_dir / "summary.json"

# Confirm the training outputs exist.
assert history_path.exists(), f"Missing Q8 history: {history_path}"
assert summary_path.exists(), f"Missing Q8 summary: {summary_path}"

# Restore the results without training again.
q8_history = pd.read_csv(history_path)

with open(summary_path, "r") as f:
    q8_summary = json.load(f)

q8_result = {
    "history": q8_history,
    "summary": q8_summary
}

# Calculate the requested Q8 answers.
q8_lowest_val_epoch = int(
    q8_history.loc[q8_history["val_loss"].idxmin(), "epoch"]
)

q8_best_score_epoch = int(
    q8_summary["best_epoch_by_official_score"]
)

q8_best_score = float(
    q8_summary["best_official_val_score"]
)

q8_gap_epoch6 = float(
    q8_history.iloc[-1]["val_loss"]
    - q8_history.iloc[-1]["train_loss"]
)

# Describe the observed six-epoch learning curve.
train_drop = float(
    q8_history["train_loss"].iloc[0]
    - q8_history["train_loss"].iloc[-1]
)

val_drop = float(
    q8_history["val_loss"].iloc[0]
    - q8_history["val_loss"].iloc[-1]
)

score_trend = float(
    q8_history["official_val_score"].iloc[-1]
    - q8_history["official_val_score"].iloc[0]
)

if train_drop > 0 and val_drop > 0 and score_trend > 0:
    q8_curve_pattern = "healthy convergence over the observed six epochs"
elif train_drop > 0 and val_drop < 0 and score_trend <= 0:
    q8_curve_pattern = "overfitting: train loss falls while validation worsens"
else:
    q8_curve_pattern = "mixed/early learning trend; inspect all six epoch curves"

# Display recovered results.
print("Q8 epoch history:")
display(q8_history.round(6))

print(f"Q8(a) Lowest validation loss epoch: {q8_lowest_val_epoch}")
print(
    f"Q8(b) Highest official validation score: "
    f"epoch {q8_best_score_epoch}, {q8_best_score:.3f}"
)
print(f"Q8(c) Generalization gap at epoch 6: {q8_gap_epoch6:.3f}")
print(
    f"Q8(d) Monitor official validation score; "
    f"keep epoch {q8_best_score_epoch}; curve: {q8_curve_pattern}."
)

print(
    "Q8 ANSWER FORMAT:",
    f"{q8_lowest_val_epoch}; "
    f"{q8_best_score_epoch}, {q8_best_score:.3f}; "
    f"{q8_gap_epoch6:.3f}; "
    f"monitor official validation score and keep epoch "
    f"{q8_best_score_epoch} — {q8_curve_pattern}."
)

print("\nRecovered checkpoint:")
print(q8_summary["checkpoint"])

Q8 epoch history:


,epoch,train_loss,val_loss,official_val_score,foreground_dice,false_positive_score,empty_accuracy,false_alarm_rate_empty_pairs_pct,lr_last_step,lr_epoch_start
0,1,0.799592,0.713958,0.281575,0.167684,0.820981,0.000000,100.000000,0.000300,0.000300
1,2,0.630878,0.572300,0.384932,0.256212,0.989550,0.076733,92.326733,0.000285,0.000300
2,3,0.519966,0.475368,0.493939,0.330593,0.990216,0.644802,35.519802,0.000244,0.000285
3,4,0.447484,0.450566,0.515467,0.364130,0.982334,0.641089,35.891089,0.000183,0.000244
4,5,0.402580,0.399098,0.540229,0.392732,0.989332,0.674505,32.549505,0.000117,0.000183
5,6,0.358103,0.371637,0.568919,0.414503,0.997180,0.793317,20.668317,0.000056,0.000117


Q8(a) Lowest validation loss epoch: 6
Q8(b) Highest official validation score: epoch 6, 0.569
Q8(c) Generalization gap at epoch 6: 0.014
Q8(d) Monitor official validation score; keep epoch 6; curve: healthy convergence over the observed six epochs.
Q8 ANSWER FORMAT: 6; 6, 0.569; 0.014; monitor official validation score and keep epoch 6 — healthy convergence over the observed six epochs.

Recovered checkpoint:
/content/drive/MyDrive/DLGenAI_Milestone4/Q8_baseline_warmup_cosine_seed42/best_checkpoint.pt


## 15. Q10 — Controlled OFAT experiments R0, R0b, R1 and R2

In [18]:
# R0: baseline (constant LR, natural sampling, AMP where CUDA exists), seed 42.
# R0b: identical to R0 except seed 7; estimates seed noise only.
# R1: R0 + warmup/cosine schedule only.
# R2: R0 + bounded square-root sampler only; sampler weights are recomputed on the 960-image training split.

q10_specs = [
    ("R0_constant_natural_seed42", 42, "constant", "natural"),
    ("R0b_constant_natural_seed7", 7, "constant", "natural"),
    ("R1_warmup_cosine_natural_seed42", 42, "warmup_cosine", "natural"),
    ("R2_constant_bounded_sqrt_seed42", 42, "constant", "bounded_sqrt"),
]
q10_results = {}
for run_name, seed, schedule, sampling in q10_specs:
    q10_results[run_name] = run_training_experiment(
        run_name=run_name, seed=seed, schedule=schedule, sampling=sampling
    )

r0_name, r0b_name, r1_name, r2_name = [x[0] for x in q10_specs]
r0 = q10_results[r0_name]["summary"]
r0b = q10_results[r0b_name]["summary"]
r1 = q10_results[r1_name]["summary"]
r2 = q10_results[r2_name]["summary"]

q10_scores = {
    "R0":float(r0["best_official_val_score"]),
    "R0b":float(r0b["best_official_val_score"]),
    "R1":float(r1["best_official_val_score"]),
    "R2":float(r2["best_official_val_score"]),
}
noise_band = abs(q10_scores["R0"] - q10_scores["R0b"])
gain_r1 = q10_scores["R1"] - q10_scores["R0"]
gain_r2 = q10_scores["R2"] - q10_scores["R0"]
r1_exceeds_noise = gain_r1 > 2.0 * noise_band
r2_exceeds_noise = gain_r2 > 2.0 * noise_band
r0_false_alarm = float(r0["best_metrics"]["false_alarm_rate_empty_pairs_pct"])
r2_false_alarm = float(r2["best_metrics"]["false_alarm_rate_empty_pairs_pct"])
if r2_false_alarm < r0_false_alarm:
    q10_prior_shift_sentence = (
        "The bounded sampler lowered the clean-pair false-alarm rate, indicating a shift "
        "toward fewer defect predictions on empty image-class pairs."
    )
elif r2_false_alarm > r0_false_alarm:
    q10_prior_shift_sentence = (
        "The bounded sampler raised the clean-pair false-alarm rate, indicating a shift "
        "toward more defect-positive predictions on empty image-class pairs."
    )
else:
    q10_prior_shift_sentence = (
        "The bounded sampler produced no measured change in the clean-pair false-alarm rate "
        "at the fixed 0.5 threshold."
    )
q10_ofat_sentence = (
    "Changing several components at once confounds their effects; one-factor-at-a-time runs "
    "isolate the scheduler and sampler effects against R0."
)

q10_table = pd.DataFrame([
    {"run":name, "best_epoch":q10_results[run_name]["summary"]["best_epoch_by_official_score"],
     "best_official_validation_score":q10_scores[name],
     "schedule":spec[2], "sampling":spec[3], "seed":spec[1]}
    for name, run_name, spec in zip(["R0","R0b","R1","R2"], [r0_name,r0b_name,r1_name,r2_name], q10_specs)
])
display(q10_table.round(6))
print(f"Q10(a) Best official validation scores R0, R0b, R1, R2: {q10_scores['R0']:.3f}, {q10_scores['R0b']:.3f}, {q10_scores['R1']:.3f}, {q10_scores['R2']:.3f}")
print(f"Q10(b) Seed-noise band: {noise_band:.3f}; isolated gain R1-R0: {gain_r1:.3f}; isolated gain R2-R0: {gain_r2:.3f}")
print(f"Q10(c) R1 > 2×noise? {'yes' if r1_exceeds_noise else 'no'}; R2 > 2×noise? {'yes' if r2_exceeds_noise else 'no'}")
print(f"Q10(d) Empty-pair false-alarm rate at threshold 0.5: R0={r0_false_alarm:.1f}%; R2={r2_false_alarm:.1f}%")
print("Q10(e)", q10_ofat_sentence)
print("Q10(f)", q10_prior_shift_sentence)
print("Q10 ANSWER FORMAT:", f"{q10_scores['R0']:.3f}, {q10_scores['R0b']:.3f}, {q10_scores['R1']:.3f}, {q10_scores['R2']:.3f}; {noise_band:.3f}, {gain_r1:.3f}, {gain_r2:.3f}; {'yes' if r1_exceeds_noise else 'no'}, {'yes' if r2_exceeds_noise else 'no'}; {r0_false_alarm:.1f}%, {r2_false_alarm:.1f}%; {q10_ofat_sentence} {q10_prior_shift_sentence}")

/tmp/ipykernel_354/2451287887.py:103: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=AMP_ENABLED)


Attempting W&B online initialisation...


Online W&B initialisation failed:
CommError Run initialization has timed out after 180.0 sec. Please try increasing the timeout with the `init_timeout` setting: `wandb.init(settings=wandb.Settings(init_timeout=120))`.
Switching to W&B offline mode...


R0_constant_natural_seed42 | epoch 1/6:   0%|          | 0/60 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7bbe5d9c9580>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1706, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1675, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
    

Validation:   0%|          | 0/15 [00:00<?, ?it/s]

... (103 outputs left out)

R2_constant_bounded_sqrt_seed42 | epoch 6: train_loss=0.33634, val_loss=0.34846, official=0.52995, lr=3.00e-04


empty_accuracy,▁▄████
epoch,▁▂▄▅▇█
false_alarm_rate_empty_pairs_pct,█▅▁▁▁▁
false_positive_score,▁▆█▆▇█
foreground_dice,▁▅▇▅▇█
lr_epoch_start,▁▁▁▁▁▁
lr_last_step,▁▁▁▁▁▁
official_val_score,▁▅▇▆▇█
train_loss,█▅▃▂▂▁
val_loss,█▅▃▃▂▁
best_epoch_by_official_score,6


,run,best_epoch,best_official_validation_score,schedule,sampling,seed
0,R0,5,0.525562,constant,natural,42
1,R0b,5,0.531330,constant,natural,7
2,R1,6,0.568919,warmup_cosine,natural,42
3,R2,6,0.529950,constant,bounded_sqrt,42


Q10(a) Best official validation scores R0, R0b, R1, R2: 0.526, 0.531, 0.569, 0.530
Q10(b) Seed-noise band: 0.006; isolated gain R1-R0: 0.043; isolated gain R2-R0: 0.004
Q10(c) R1 > 2×noise? yes; R2 > 2×noise? no
Q10(d) Empty-pair false-alarm rate at threshold 0.5: R0=36.8%; R2=26.1%
Q10(e) Changing several components at once confounds their effects; one-factor-at-a-time runs isolate the scheduler and sampler effects against R0.
Q10(f) The bounded sampler lowered the clean-pair false-alarm rate, indicating a shift toward fewer defect predictions on empty image-class pairs.
Q10 ANSWER FORMAT: 0.526, 0.531, 0.569, 0.530; 0.006, 0.043, 0.004; yes, no; 36.8%, 26.1%; Changing several components at once confounds their effects; one-factor-at-a-time runs isolate the scheduler and sampler effects against R0. The bounded sampler lowered the clean-pair false-alarm rate, indicating a shift toward fewer defect predictions on empty image-class pairs.


## 16. Final answer sheet and saved artifacts

In [19]:
answer_sheet = {
    "Q1": f"{clean_count}; {positive_by_class['defect_a']}, {positive_by_class['defect_b']}, {positive_by_class['defect_c']}, {positive_by_class['defect_d']}; {q1_unique_combinations}; {q1_largest_size}, {q1_smallest_size}",
    "Q2": f"{b_positive_percentage:.4f}%; {b_mean_positive_area:.1f}; {b_pos_weight_rounded}",
    "Q3": f"{100*clean_sampler_fraction:.2f}%, {100*clean_natural_fraction:.2f}%; {b_sampler_expected:.2f}, {b_natural_expected:.2f}; {b_enrichment:.2f}x; {cap_group_count}",
    "Q4": f"{q4_lrs[15]:.2e}, {q4_lrs[270]:.2e}, {q4_lrs[400]:.2e}; {q4_bug_lr:.2e}; scheduler must step per optimizer update, not per epoch.",
    "Q5": f"{q5_stop}, {q5_restore_epoch}, {q5_restore_score:.3f}; {'yes' if q5_zero_stop else 'no'}, {q5_zero_score:.3f}",
    "Q6": f"{state_mb:.0f} MB; {fp32_batch16_mb:.0f}, {amp_batch16_mb:.0f} MB; {max_batch_fp32}, {max_batch_amp}; {effective_batch}; {micro_batches_per_epoch}, {optimizer_steps_per_epoch}; {accumulation}; {warmup_steps_2_epochs}",
    "Q7A": f"{empty_pairs}; {baseline_naive_dice:.4f}; {baseline_official_score:.4f}",
    "Q7B": f"{toy_official_score:.4f}, {toy_naive_dice:.4f}; {naive_metric_verdict}; {baseline_naive_dice:.4f}",
    "Q8": f"{q8_lowest_val_epoch}; {q8_best_score_epoch}, {q8_best_score:.3f}; {q8_gap_epoch6:.3f}; monitor official score, keep epoch {q8_best_score_epoch}; {q8_curve_pattern}",
    "Q9": f"{bin_counts['1-100']}, {bin_counts['101-500']}, {bin_counts['501+']}; {large_pixel_percentage:.2f}%; {missed_total:.2f}, {most_missed_bin}; pair-count and pixel-count rankings differ because defect sizes differ.",
    "Q10": f"{q10_scores['R0']:.3f}, {q10_scores['R0b']:.3f}, {q10_scores['R1']:.3f}, {q10_scores['R2']:.3f}; {noise_band:.3f}, {gain_r1:.3f}, {gain_r2:.3f}; {'yes' if r1_exceeds_noise else 'no'}, {'yes' if r2_exceeds_noise else 'no'}; {r0_false_alarm:.1f}%, {r2_false_alarm:.1f}%; {q10_ofat_sentence} {q10_prior_shift_sentence}"
}

print("="*80)
print("DLGenAI PROJECT 26T3 — MILESTONE 4 ANSWER SHEET")
print("="*80)
for key, value in answer_sheet.items():
    print(f"{key}: {value}")

report = {
    "milestone": "DLGenAI Project 26T3 - Milestone 4",
    "working_subset_images": 1200,
    "train_images": len(train_ids), "validation_images": len(val_ids),
    "split_seed": 42, "classes": CLASSES,
    "q1": {"clean_images":clean_count,"positive_images_by_class":positive_by_class,
           "raw_unique_combinations":q1_unique_combinations,"raw_largest_group":q1_largest_size,
           "raw_smallest_group":q1_smallest_size,
           "merged_largest_group":merged_largest_size,"merged_smallest_group":merged_smallest_size,
           "rare_groups_merged":sorted(rare_groups)},
    "q2": {"positive_pixel_percentage":b_positive_percentage,"mean_positive_area":b_mean_positive_area,
           "pos_weight_candidate":b_pos_weight_rounded},
    "q3": {"sampler_clean_pct":100*clean_sampler_fraction,"natural_clean_pct":100*clean_natural_fraction,
           "sampler_defect_b_draws":b_sampler_expected,"natural_defect_b_draws":b_natural_expected,
           "enrichment":b_enrichment,"groups_at_cap":cap_group_count},
    "q4": {"lr_steps":q4_lrs,"buggy_epoch8_lr":q4_bug_lr},
    "q5": {"stop_epoch":q5_stop,"restored_epoch":q5_restore_epoch,"restored_score":q5_restore_score,
           "zero_delta_stops":q5_zero_stop,"zero_delta_best_epoch":q5_zero_epoch,"zero_delta_best_score":q5_zero_score},
    "q6": {"state_mb":state_mb,"fp32_batch16_mb":fp32_batch16_mb,"amp_batch16_mb":amp_batch16_mb,
           "max_batch_fp32":max_batch_fp32,"max_batch_amp":max_batch_amp,"effective_batch":effective_batch,
           "micro_batches_per_epoch":micro_batches_per_epoch,"optimizer_steps_per_epoch":optimizer_steps_per_epoch,
           "loss_divisor":accumulation,"two_epoch_warmup_steps":warmup_steps_2_epochs},
    "q7": {"empty_pairs":empty_pairs,"baseline_naive_dice":baseline_naive_dice,
           "baseline_official_score":baseline_official_score,"toy_official_score":toy_official_score,
           "toy_naive_dice":toy_naive_dice,"verdict":naive_metric_verdict},
    "q8": q8_summary,
    "q9": {"bin_counts":bin_counts,"large_pixel_percentage":large_pixel_percentage,
           "expected_missed_pairs":missed_total,"largest_missed_bin":most_missed_bin},
    "q10": {"scores":q10_scores,"noise_band":noise_band,"gain_R1":gain_r1,"gain_R2":gain_r2,
            "R1_exceeds_2x_noise":r1_exceeds_noise,"R2_exceeds_2x_noise":r2_exceeds_noise,
            "false_alarm_R0_pct":r0_false_alarm,"false_alarm_R2_pct":r2_false_alarm},
    "answer_sheet":answer_sheet,
    "wandb_entity":WANDB_ENTITY,"wandb_project":WANDB_PROJECT,
    "device":str(DEVICE),"amp_enabled":AMP_ENABLED,
}
report_path = ARTIFACT_DIR / "milestone4_final_report.json"
with open(report_path,"w") as f:
    json.dump(report,f,indent=2,default=lambda x: float(x) if isinstance(x,np.floating) else int(x) if isinstance(x,np.integer) else x)
answer_path = ARTIFACT_DIR / "milestone4_answer_sheet.txt"
with open(answer_path,"w") as f:
    for key,value in answer_sheet.items(): f.write(f"{key}: {value}\n")
q8_history.to_csv(ARTIFACT_DIR / "Q8_baseline_history.csv",index=False)
q10_table.to_csv(ARTIFACT_DIR / "Q10_OFAT_summary.csv",index=False)
print("\nSaved report:",report_path)
print("Saved answer sheet:",answer_path)
print("All training runs and metrics are persisted in:",ARTIFACT_DIR)

DLGenAI PROJECT 26T3 — MILESTONE 4 ANSWER SHEET
Q1: 578; 107, 35, 438, 181; 15; 578, 1
Q2: 0.0160%; 561.5; 6251
Q3: 28.81%, 48.17%; 104.69, 35.00; 2.99x; 10
Q4: 7.50e-05, 1.50e-04, 2.61e-05; 4.00e-05; scheduler must step per optimizer update, not per epoch.
Q5: 10, 7, 0.671; no, 0.673
Q6: 400 MB; 2800, 1600 MB; 20, 40; 32; 120, 30; 4; 60
Q7A: 4039; 0.8415; 0.3000
Q7B: 0.6398, 0.8400; empty-aware mean Dice ranks the toy model below the all-empty baseline; 0.8415
Q8: 6; 6, 0.569; 0.014; monitor official score, keep epoch 6; healthy convergence over the observed six epochs
Q9: 5, 99, 657; 99.24%; 66.05, 501+; pair-count and pixel-count rankings differ because defect sizes differ.
Q10: 0.526, 0.531, 0.569, 0.530; 0.006, 0.043, 0.004; yes, no; 36.8%, 26.1%; Changing several components at once confounds their effects; one-factor-at-a-time runs isolate the scheduler and sampler effects against R0. The bounded sampler lowered the clean-pair false-alarm rate, indicating a shift toward fewer def